# JavaScript — Async

## LESSON 42 — Promises, async & await

Some things don't finish immediately: reading a file, waiting a second, calling an API. JavaScript doesn't stand still waiting — it moves on, and comes back when the result is ready.

### The order surprises everyone once

```js
console.log("first");
setTimeout(() => console.log("second"), 1000);
console.log("third");

// prints: first, third, second
```

> In a notebook you may not see `second` under the cell that produced it. The cell finishes long before the timer fires, so the late output can appear under whichever cell you run next. That is the delay doing its job, not a mistake.

### A Promise

A **Promise** is a value that isn't here yet. It will either **resolve** (success) or **reject** (failure).

```js
const wait = (ms) =>
  new Promise((resolve) => setTimeout(resolve, ms));
```

### await — the readable way

`await` pauses until the Promise settles, then gives you the **value inside** it.

```js
const data = await fetchUser();   // data is the user, not a Promise
```

`await` only works inside a function marked `async`. Notebook cells are the exception: at the outermost level of a cell you can `await` directly, without wrapping it in a function.

```js
async function loadUser() {
  const user = await fetchUser();
  return user;
}
```

**An `async` function always returns a Promise**, even when you return a plain value.

### Handling failure

```js
try {
  const user = await fetchUser();
  console.log(user);
} catch (error) {
  console.log("Something went wrong:", error.message);
}
```

### Key notes

- **Forgetting `await` gives you the Promise, not the data.** If you print something like `Promise { <pending> }`, that's the missing `await`.
- `await` inside a normal (non-`async`) function is a syntax error.
- Calling an `async` function does **not** pause your code — only `await` does.
- Use `try/catch` around `await`: a rejected Promise throws.

### Example

In [ ]:
// 1. Order of execution
console.log("first");
setTimeout(() => console.log("second (one second later)"), 1000);
console.log("third");

// 2. A promise that resolves after a delay
const exampleWait = (ms) => new Promise((resolve) => setTimeout(resolve, ms));

// 3. A fake API call
async function exampleFetchUser() {
  await exampleWait(300);
  return { name: "Sam", role: "student" };
}

// Without await: a Promise, not the data
console.log(exampleFetchUser());

// With await: the data itself
const exampleUser = await exampleFetchUser();
console.log(exampleUser);
console.log(exampleUser.name);

// 4. Failure handled with try/catch
async function exampleFailingCall() {
  await exampleWait(100);
  throw new Error("server unreachable");
}

try {
  await exampleFailingCall();
} catch (error) {
  console.log("Caught:", error.message);
}

### Exercise

1. Write `delay(ms)` that returns a Promise resolving after `ms` milliseconds.
2. Write an `async` function `getProduct()` that waits 200ms and then returns
   `{ name: "Laptop", price: 1200 }`.
3. Call it **with** `await`, store the result in `product`, and print the name.
4. Then print `getProduct()` **without** `await` and look at what you get instead.

_Don't open `solutions.ipynb` until you've actually tried._

In [ ]:
// Your code here
//1
const delay=(ms)=> new Promise((res)=>setTimeout(res,ms));
//2
async function getProduct(){
    await delay(200);
    return {name:"laptop",price:1200};
}
//3
let product=await getProduct();
console.log(product);
//4
product= getProduct();
console.log(product);


### Mini challenge

1. Write an `async` function `login(username)` that waits 200ms, then:
   - throws `new Error("unknown user")` if the username is not `"sam"`
   - otherwise returns `"Welcome, sam"`
2. Call it with `"sam"` inside a `try/catch` and print the result.
3. Call it with `"alex"` inside a `try/catch` and print the error message.

Both calls must run without crashing the cell.

In [ ]:
// Your code here
async function login(username){
    delay(200);
    if(username!=="Sam") throw new Error("unknown user");
    return "Welcome Sam";
}
try{
    await login("Sam");
    await login("Alex");

}catch(error){
    console.log("Error login: ", error.message);
}


## LESSON 43 — Promise combinators and timers

### The other way to read a promise

`await` is newer syntax for something that already existed: `.then()`. Both read the value out of the same promise — but only `await` pauses. `.then()` registers what to do later and lets the code carry straight on.

```js
// same promise, same value, different timing
const data = await getData();   // waits here, then continues

getData().then((data) => console.log(data));
console.log("this runs before the value arrives");
```

| with await | with then |
|---|---|
| `const x = await p;` (pauses) | `p.then((x) => ...)` (does not pause) |
| `try { } catch (e) { }` | `.catch((e) => ...)` |
| `finally { }` | `.finally(() => ...)` |

Prefer `await`: it reads top to bottom, and its errors go through the same `try`/`catch` as everything else. Learn `.then` because you will read it constantly in other people's code — and because a chain like `fetch(url).then(r => r.json())` is genuinely shorter.

### Waiting for several things at once

Awaiting one after another makes them queue up. When they do not depend on each other, that wastes time.

```js
// 2 seconds: one after the other
const a = await slow();
const b = await slow();

// 1 second: at the same time
const [a, b] = await Promise.all([slow(), slow()]);
```

| combinator | resolves when | if one fails |
|---|---|---|
| `Promise.all` | **all** succeed | the whole thing rejects at once |
| `Promise.allSettled` | all finish, either way | nothing rejects; you inspect each result |
| `Promise.race` | the **first** one settles | that first result wins, success or failure |
| `Promise.any` | the first one **succeeds** | rejects only if they all fail |

`allSettled` gives you an array of `{ status, value }` or `{ status, reason }` — the right choice when a partial result is still useful.

### Timers

```js
const id = setTimeout(() => console.log("once"), 1000);
clearTimeout(id);

const ticker = setInterval(() => console.log("tick"), 1000);
clearInterval(ticker);          // otherwise it runs forever
```

`setInterval` repeats until you stop it. Keep the id it returns — without it you have no way to stop it, and a page that fetches on a forgotten interval keeps fetching after the user has moved on.

### Key notes

- **`Promise.all` fails all-or-nothing.** One rejection discards the results of the others. Use `allSettled` when partial success is still worth having.
- **`Promise.all` does not start the work.** The promises are already running when you build the array; `all` only waits. Passing `[slow(), slow()]` starts both immediately, which is the point.
- **Always keep the id `setInterval` returns.** No id, no way to stop it.
- A timer's delay is a **minimum**, not a guarantee. `setTimeout(fn, 0)` still runs after the current code finishes.

### Example

In [ ]:
function exampleDelay(ms, value) {
  return new Promise((resolve) => setTimeout(() => resolve(value), ms));
}

function exampleFail(ms) {
  return new Promise((_, reject) => setTimeout(() => reject(new Error("nope")), ms));
}

exampleDelay(50, "with then").then((value) => console.log(value));

const exampleStart = Date.now();
const exampleBoth = await Promise.all([exampleDelay(100, "a"), exampleDelay(100, "b")]);
console.log(exampleBoth, `in about ${Date.now() - exampleStart}ms`);

const exampleSettled = await Promise.allSettled([exampleDelay(10, "ok"), exampleFail(10)]);
console.log(exampleSettled.map((result) => result.status));

console.log(await Promise.race([exampleDelay(10, "fast"), exampleDelay(200, "slow")]));
console.log(await Promise.any([exampleFail(10), exampleDelay(20, "survivor")]));

let exampleCount = 0;
const exampleTicker = setInterval(() => {
  exampleCount += 1;
  console.log("tick", exampleCount);
  if (exampleCount === 3) clearInterval(exampleTicker);
}, 30);

### Exercise

Use this helper:

```js
function delayValue(ms, value) {
  return new Promise((resolve) => setTimeout(() => resolve(value), ms));
}
```

It has its own name because the `delay(ms)` you wrote for LESSON 42 is still defined in this notebook, and a second `delay` would clash with it.

1. Call `delayValue(100, "hello")` with `.then()` and print the result.
2. Call the same thing with `await` and print it.
3. Await `delayValue(100, "a")` and `delayValue(100, "b")` **at the same time** with `Promise.all`, and print both.
4. Print how many milliseconds step 3 took, using `Date.now()` before and after.
5. Print the winner of a `Promise.race` between a 20ms and a 200ms delay.

_Don't open `solutions.ipynb` until you've actually tried._

In [ ]:
// Your code here
function delayValue(ms, value) {
  return new Promise((resolve) => setTimeout(() => resolve(value), ms));
}
//1

delayValue(100,"Hello").then((result)=>console.log(result));
//2
await delayValue(100,"Hello").then((result)=>console.log(result));
//3
let bothfun=await Promise.all([delayValue(100,"a"),delayValue(100,"b")]);
console.log(bothfun);
//4
const start=Date.now();
let bothfun2=await Promise.all([delayValue(100,"a"),delayValue(100,"b")]);
console.log(bothfun2,`work in ${Date.now()-start}ms`);
//5
console.log(await Promise.race([delayValue(20,"fast"),delayValue(200,"slow")]));



### Mini challenge

1. Write `unreliable(shouldFail)` returning a promise that resolves to `"ok"` after 20ms, or rejects with an `Error` when `shouldFail` is `true`.
2. Run three of them — one failing, two succeeding — through `Promise.allSettled`, and print the status of each.
3. Do the same through `Promise.all` inside a `try`/`catch`. Print the error message, and add a comment saying what you no longer have access to.
4. Write a counter with `setInterval` that prints 1, 2, 3 and then stops itself. It must not keep running afterwards.

In [ ]:
// Your code here
//1
function unreliable(shouldFail){
    return new Promise((resolve,reject)=>{
        setTimeout(() => {
            if(shouldFail) return reject(new Error("Failled"));
            else resolve("ok");
        }, 20);
    });
}
//2
let result=await Promise.allSettled([unreliable(false),unreliable(true),unreliable(true)]);
console.log( result.map((item)=>item.status));
//3
 
try{
    await Promise.all([unreliable(false),unreliable(true),unreliable(true)]);
}catch(error)
{
    console.log("All rejected:",error.message);
}
//4 

let count=0;
const timer=
        await setInterval(() => {
        console.log(count+1);
        count++;

        if(count===3) return clearInterval(timer);
       
    }, 1000);
console.log("Finished timer");